
feature engineering for BodyFat.csv (extended dataset)

adds a 'NavyBodyFat' column using the US Navy circumference method,
and drops the 'Original' y/n column.

US Navy formula (imperial units: inches):
    Men:   %BF = 86.010 * log10(Abdomen - Neck) - 70.041 * log10(Height) + 36.76
    Women: %BF = 163.205 * log10(Waist + Hip - Neck) - 97.684 * log10(Height) - 78.387


In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("BodyFat.csv")

df = df.drop(columns="Original")

df["Height"] = df["Height"] *100 # m to cm
df["Height_in"] = df["Height"] / 2.54
df["Abdomen_in"] = df["Abdomen"] / 2.54
df["Neck_in"] = df["Neck"] / 2.54
df["Hip_in"] = df["Hip"] / 2.54 


def navy_bodyfat_male(abdomen, neck, height):
    return 86.010 * np.log10(abdomen - neck) - 70.041 * np.log10(height) + 36.76


def navy_bodyfat_female(waist, hip, neck, height):
    return 163.205 * np.log10(waist + hip - neck) - 97.684 * np.log10(height) - 78.387


def compute_navy(row):
    is_male = str(row["Sex"]).strip().upper()=="M"
    if is_male:
        return navy_bodyfat_male(row["Abdomen_in"], row["Neck_in"], row["Height_in"])
    else:
        return navy_bodyfat_female(row["Abdomen_in"], row["Hip_in"], row["Neck_in"], row["Height_in"])



df["NavyBodyFat"] = df.apply(compute_navy, axis=1)
df=df.drop(columns=["Hip_in","Neck_in","Abdomen_in","Height_in"])

df.to_csv("BodyFat_engineered.csv", index=False)


Feature usefulness check for BodyFat_engineered.csv

1. Correlation of every numeric feature with BodyFat (target) - all rows
2. Same, split by Sex (M / F) separately

This is a quick screening step - high correlation = likely useful,
low correlation overall but high within one sex = sex-specific signal.


In [ ]:
import pandas as pd
df = pd.read_csv("BodyFat_engineered.csv")

EXCLUDE = ["BodyFat", "Density", "NavyBodyFat", "Sex"]

feature_cols = [c for c in df.columns if c not in EXCLUDE and pd.api.types.is_numeric_dtype(df[c])]


def correlation_report(data: pd.DataFrame):
    
    corrs = data[feature_cols].corrwith(data["BodyFat"]).sort_values(key=abs, ascending=False)
    for feat, val in corrs.items():
        print(f"  {feat:<15} {val:+.3f}")

print("all rows")
correlation_report(df)

male = df[df["Sex"].str.strip().str.upper() == "M"]
female = df[df["Sex"].str.strip().str.upper() == "F"]

male_corr = male[feature_cols].corrwith(male["BodyFat"])
female_corr = female[feature_cols].corrwith(female["BodyFat"])

  Hip             +0.588
  Knee            +0.367
  Abdomen         +0.357
  Weight          +0.346
  Thigh           +0.331
  Chest           +0.310
  Biceps          +0.258
  Ankle           +0.183
  Height          -0.151
  Neck            +0.122
  Forearm         +0.106
  Wrist           +0.089
  Age             +0.016
